In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Change only these if your Unity Catalog names are different
CATALOG = f"formula1_{env}"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

def bronze_table(name):
    return f"{CATALOG}.{BRONZE}.{name}"

def silver_table(name):
    return f"{CATALOG}.{SILVER}.{name}"

In [0]:
constructors=spark.table(bronze_table("constructors"))
constructors.printSchema()
print("Bronze rows:",constructors.count())

## 1. NULL + duplicate checks

In [0]:
display(constructors.filter(F.col("constructor_id").isNull()))
display(constructors.groupBy("constructor_id").count().filter(F.col("count")>1))

## 2. String operations

In [0]:
constructors_clean=(
    constructors
    .filter(F.col("constructor_id").isNotNull())
    .dropDuplicates(["constructor_id"])
    .withColumn("constructor_ref_clean",F.lower(F.trim("constructor_ref")))
    .withColumn("constructor_name_clean",F.trim("name"))
    .withColumn("nationality_clean",F.upper(F.trim("nationality")))
    .withColumn("ref_length",F.length("constructor_ref_clean"))
    .withColumn("ref_prefix",F.substring("constructor_ref_clean",1,3))
    .withColumn("ref_parts",F.split("constructor_ref_clean","_"))
)
display(constructors_clean.limit(20))

## 3. contains / startsWith / endsWith

In [0]:
display(constructors_clean.filter(F.col("constructor_ref_clean").contains("a"))
                  .select("constructor_ref_clean","constructor_name_clean").limit(20))
display(constructors_clean.filter(F.col("constructor_ref_clean").startswith("a"))
                  .select("constructor_ref_clean","constructor_name_clean").limit(20))
display(constructors_clean.filter(F.col("constructor_ref_clean").endswith("a"))
                  .select("constructor_ref_clean","constructor_name_clean").limit(20))

## 4. concat + CASE

In [0]:
constructors_clean=(
    constructors_clean
    .withColumn(
        "constructor_label",
        F.concat_ws(" - ","constructor_name_clean","nationality_clean")
    )
    .withColumn(
        "constructor_ref_category",
        F.when(F.col("ref_length")<=5,"Short")
         .when(F.col("ref_length")<=10,"Medium")
         .otherwise("Long")
    )
)
display(constructors_clean.select(
    "constructor_id","constructor_label",
    "ref_length","constructor_ref_category"
).limit(20))

## 5. GroupBy + OrderBy

In [0]:
display(
    constructors_clean.groupBy("nationality_clean").count()
                      .orderBy(F.desc("count"))
)
display(constructors_clean.orderBy(F.asc("constructor_name_clean")).limit(20))

## 6. Write Silver

In [0]:
constructors_silver=constructors_clean.select(
    "constructor_id","constructor_ref_clean","constructor_name_clean",
    "nationality_clean","ref_length","ref_prefix","ref_parts",
    "constructor_label","constructor_ref_category",
    "ingestion_timestamp","ingestion_date"
)

constructors_silver.printSchema()

(
    constructors_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(silver_table("constructors"))
)
print("Created:",silver_table("constructors"))

In [0]:
silver=spark.table(silver_table("constructors"))
print("Silver rows:",silver.count())
display(silver.limit(20))

In [0]:
# for table in [
#     "circuits", 
#     "races",
#     "constructors",
#     "drivers",
#     "results",
#     "pit_stops",
#     "lap_times",
#     "qualifying"
# ]:
#     print(f"\n===== {table} =====")
#     display(
#         spark.table(f"formula1_dev.silver.{table}").printSchema()
#     )